# 3.01 - Visualizing Airborne Events

Plotting all haunted places flagged as **{Plane_Crash, Flying_Object, Electronic_Malfunction}**

- All Flight and airport data data is from notebooks [2.01]

In [245]:
# System Path #
import os
import sys 

# Add dsci_550_a1 to base path. Lets you project functions #
parent_dir = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.append(parent_dir)

# Pandas #
import pandas as pd
import time
import re
import json 


# Runtime #
import time
from tqdm import tqdm 

# Iterators #
import collections
import ast
import random

# Flight Trajectory Functions #
from dsci_550_a1.flightFunctions import *

# Plotting #
import plotly.graph_objects as go


## Load Data

In [ ]:
## Load haunted places with added features
df_american_routes = pd.read_csv("../data/joined_datasets/american_routes.tsv", sep = "\t")

## Our Airports
df_american_airports = pd.read_csv("../data/joined_datasets/american_airports.tsv", sep = "\t")

## Open Flights Dataset
df_haunted_places = pd.read_csv("../data/processed/haunted_places_features_added.tab", sep = "\t")

## Pandas stores nested dicts and lists as strings ##
## This converts them back to lists and dicts ##

df_american_routes["Flight_Path"] = df_american_routes["Flight_Path"].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)
df_american_airports["Airport_Radius"] = df_american_airports["Airport_Radius"].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)



In [248]:

df_american_routes = pd.read_csv("../data/joined_datasets/american_routes.tsv", sep = "\t")
df_american_airports = pd.read_csv("../data/joined_datasets/american_airports.tsv", sep = "\t")
df_haunted_places = pd.read_csv("../data/processed/haunted_places_features_added.tab", sep = "\t")


df_american_routes["Flight_Path"] = df_american_routes["Flight_Path"].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)
df_american_airports["Airport_Radius"] = df_american_airports["Airport_Radius"].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)

## Flight Intersection Data
with open("../data/processed/flight_proximity_data.json") as f:
    flight_intersection_data = json.load(f)
    
## Airport Intersection Data
with open("../data/processed/airport_proximity_data.json") as f:
    airport_intersection_data = json.load(f)

In [ ]:
####################################################################################################
## Flag relevant values and specify column

col = 'Event_Type'
flagged_values = ["Plane_Crash", "Flying_Object", "Electronic_Malfunction"]

## Filter Df
df_haunted_places_filtered = df_haunted_places[df_haunted_places[col].apply(lambda x: any(event in x for event in flagged_values))].drop_duplicates()

## Store entry IDs
# IDs are used to link flight and airport intersection data to haunted place
haunted_places_filtered_idxs = df_haunted_places_filtered.index.tolist()

#############################################
## Assign Flagged values in hierarchy
# plotly does not allow for multi-legend entries. Because of this each haunted place gets a single legend name. 
# For entries that have more than one flagged value, we assign it the first flag it has in the order of "flagged_values"

for idx in haunted_places_filtered_idxs:
    x = df_haunted_places_filtered.loc[idx, f'{col}']
    while x not in flagged_values:
        for flag in flagged_values:
            if flag in x:
                df_haunted_places_filtered.loc[idx, f'{col}'] = flag
                x = flag
                break

####################################################################################################
## Filter df_american_airports and df_american_routes


# Init lists to store flight route and airport 
route_indices = []
airport_Iata_codes = []
airport_idxs =[]

## Add all airports and routes that correspond to filtered haunted places ##
for idx in haunted_places_filtered_idxs:
    idx = str(idx) 

    # Add all route indices to "route_indicies" that intersect with haunted place 
    [route_indices.append(route['Route_ID']) for route in  flight_intersection_data[idx]["Routes"] if route['Route_ID'] not in route_indices]

    # Add source and destination airport code to "airport_Iata_codes" if not already added
    [airport_Iata_codes.append(route['Source_Airport']) for route in  flight_intersection_data[idx]["Routes"] if route['Source_Airport'] not in airport_Iata_codes]
    [airport_Iata_codes.append(route['Dest_Airport']) for route in  flight_intersection_data[idx]["Routes"] if route['Dest_Airport'] not in airport_Iata_codes]

    # Add airport indices that intersect with haunted place
    [airport_idxs.append(airport["Airport_ID"]) for airport in  airport_intersection_data[idx]["Airports"] if airport['Airport_ID'] not in airport_idxs]


# Locate IATA codes in "airport_Iata_codes" and add corresponding index from airports_df if not already included
for code in airport_Iata_codes:
    extra_airport_indicies = df_american_airports[df_american_airports['Iata_Code'].apply(lambda x: x in airport_Iata_codes)].index.tolist()
    [airport_idxs.append(idx) for idx in extra_airport_indicies if idx not in airport_idxs]


## Fliter routes and airports

routes_filtered = df_american_routes.loc[route_indices, ["Flight_Path"]]
airports_filtered = df_american_airports.loc[airport_idxs]


In [ ]:
import textwrap

##########################################################################################
## Haunted Places Traces ##

# Initialize lists to store traces
all_traces = []
haunting_traces_idx = [] 
airports_traces_idx = [] 
flights_traces_idx = []


unique_flags = df_haunted_places_filtered[f'{col}'].unique().tolist()

plot_colors = ['rgb(127,255,0)', 'rgb(218,112,214)', 'rgb(138,43,226)',]

# plot_colors = ['rgb{240,163,255}','rgb{0,117,220}','rgb{153,63,0}','rgb{76,0,92}','rgb{25,25,25}','rgb{0,92,49}','rgb{43,206,72}','rgb{255,204,153}','rgb{128,128,128}','rgb{148,255,181}','rgb{143,124,0}','rgb{157,204,0}','rgb{194,0,136}','rgb{0,51,128}','rgb{255,164,5}','rgb{255,168,187}','rgb{66,102,0}','rgb{255,0,16}','rgb{94,241,242}','rgb{0,153,143}','rgb{224,255,102}','rgb{116,10,255}','rgb{153,0,0}','rgb{255,255,128}','rgb{255,255,0}','rgb{255,80,5}']

## Add Trace Depending on Haunted Type ##
for i, flag in enumerate(flagged_values):
    haunted_places_to_plot = df_haunted_places_filtered.loc[df_haunted_places_filtered[f'{col}'] == flag]
    haunted_places_to_plot['Formatted_Description'] = haunted_places_to_plot['Description'].apply(
    lambda x: "<br>".join(textwrap.wrap(x, width=50))
)
    trace = (go.Scattergeo(
        locationmode = 'USA-states',
        lon = haunted_places_to_plot['Longitude'],
        lat = haunted_places_to_plot['Latitude'],
        hoverinfo = 'text',
        text = haunted_places_to_plot.apply(lambda row: f"Haunting Type: {row[f'{col}']}<br>index:{row['Haunted_Places_Id']} | Location: {row['Location']}<br># Intersecting Flights: {row['Flight_Intersection_Count']} | # Nearby Airports: {row['Aerodrome_Count']}<br>Description: {row['Formatted_Description']}", axis=1),
        mode = 'markers',
        showlegend = True, 
        marker = dict(
            size = 4,
            color = plot_colors[i],
            opacity = 0.75
            ),
            name = flag,
            visible = False
        )
    )
    # Add trace
    all_traces.append(trace)
    # Store index of trace in haunting_traces_idx
    haunting_traces_idx.append(len(all_traces) - 1)


## Unpack Flight Path Coords ##
lats_plot, lons_plot = [] , []
for row in routes_filtered.itertuples(index = False):   

    lats, lons = zip(*row.Flight_Path)
    lats, lons = list(lats), list(lons)

    lats_plot.extend(lats + [None])
    lons_plot.extend(lons + [None])

## Add Flight Path Trace ##
trace = (go.Scattergeo(
    lon= lons_plot,
    lat= lats_plot,
    mode='lines',
    line=dict(width=.5, color='red'),
    opacity = 0.2, 
    hoverinfo = 'skip', 
    name = "Flights",
    visible = False
))
# Add Flight Trace to all traces
all_traces.append(trace)
# Store index of flight trace in flight_traces_idx
flights_traces_idx.append(len(all_traces) - 1)

## Add Airports ##

airport_types = airports_filtered['Type'].unique().tolist()

airport_plot_colors = {
'heliport' :        "rgb(100,151,177)" ,
 'seaplane_base': 	"rgb(179,205,224)",
 'balloonport' : 	"rgb(179,205,224)",
 'small_airport' :  "rgb(0,91,150)"  ,
 'medium_airport' :	"rgb(3,57,108)",
 'large_airport':   "rgb(1,31,75)"
}

airport_proximity_dict = {
    "large_airport" : 55560,    # 30 nautical miles
    "medium_airport" : 9260,    # 5 nautical miles
    "small_airport" : 5556,     # 3 nautical miles
    "heliport":  2778,          # 1.5 nautical miles
    "seaplane_base" : 5556,     # 3 nautical miles
    "balloonport" : 5556        # 3 nautical miles
}

## Plot airports ##
for airport_type in airport_types:

    ## Airport Marker Trace ##
    airports_to_plot = airports_filtered.loc[airports_filtered['Type'] == airport_type]

    airports_trace = (go.Scattergeo(
    locationmode = 'USA-states',
    lon = airports_to_plot['Longitude_Deg'],
    lat = airports_to_plot['Latitude_Deg'],
    hoverinfo = 'text',
    text = airports_to_plot.apply(lambda row: f"IATA Code: {row['Iata_Code']}<br>Name: {row['Name']}", axis=1),
    # text = airports_to_plot['Iata_Code'],
    mode = 'markers',
    marker = dict(
        size = 2,
        color = airport_plot_colors[airport_type],
        opacity = 1
        ),
        name = airport_type,
        visible = False
        ))
    
    ## Airport Radius Trace ##
    lons_plot = [] 
    lats_plot = []
    for airport in airports_to_plot.itertuples():
        ## Unpack Coords ##
        lats, lons = zip(*airport.Airport_Radius)
        lats, lons = list(lats), list(lons)

        lats_plot.extend(lats + [None])
        lons_plot.extend(lons + [None])
    
    airport_radii = (go.Scattergeo(
    locationmode = 'USA-states',
    lon = lons_plot,
    lat = lats_plot,
    hoverinfo = 'skip',
    mode = 'lines',
    line = dict(
        width = 1,
        color = airport_plot_colors[airport_type],
        dash = 'dot'
        ),
        name = airport_type,
        visible = False
        ))
    
    # Add airport trace to plot
    all_traces.append(airports_trace)
    # Add airport radius to plot
    all_traces.append(airport_radii)
    # Store index of aiport and airport radius in airport_traces_idx
    airports_traces_idx.append((len(all_traces) - 2 , len(all_traces) - 1))



##########################################################################################
## Menus ##

## Add Interactive Buttons for Haunts ##

button_haunts = [
        {
            "method": "restyle",
            # When toggled on, checkbox shows already visible traces + haunted place specified in box

            # "args": [{"visible" : True}, 
            #         [
            #          (current or (i == haunting_traces_idx[j])) 
            #          for i, current in enumerate([trace.visible for trace in fig.data])
            #         ]
            #         ],

            "args" : [{"visible" : True}, [i for i, x in enumerate(all_traces) if x.name == flagged_value]],
            # When toggled off, checkbox removes haunted trace
            "args2" : [{'visible':'legendonly'},[i for i,x in enumerate(all_traces) if x.name == flagged_value]],
            # "args2" : ["visible", [trace.visible for trace in fig.data]], 
            "label": flagged_value,
            "visible" : True, 

        }
        for flagged_value in flagged_values
    ]

all_haunts_button = {
            "method": "restyle",
            # When toggled on, checkbox shows already visible traces + haunted place specified in box
            "args" : [{"visible" : True}, [i for i, x in enumerate(all_traces) if x.name in flagged_values]],
            # When toggled off, checkbox removes haunted trace
            "args2" : [{'visible':'legendonly'},[i for i,x in enumerate(all_traces) if x.name in flagged_values]],
            # "args2" : ["visible", [trace.visible for trace in fig.data]], 
            "label": "Toggle All",
            "visible" : True, 

        }
button_haunts.append(all_haunts_button) 


## Add Interactive Buttons for airports ##

buttons_airports = [
        {
            "method": "restyle",
            # When toggled on, checkbox shows already visible traces + haunted place specified in box
            "args" : [{"visible" : True}, [i for i, x in enumerate(all_traces) if x.name == airport_type]],
            # When toggled off, haunted trace is removed from graph. Only legend is visible
            "args2" : [{'visible':'legendonly'},[i for i,x in enumerate(all_traces) if x.name == airport_type]],

            "label": airport_type,
            "visible" : True, 
        }
        for airport_type in airport_types
    ]

## Show all airports button ##
all_airports_button = {
            "method": "restyle",
            # When toggled on, checkbox shows already visible traces + haunted place specified in box
            "args" : [{"visible" : True}, [i for i, x in enumerate(all_traces) if x.name in airport_types]],
            # When toggled off, checkbox removes haunted trace
            "args2" : [{'visible':'legendonly'},[i for i,x in enumerate(all_traces) if x.name in airport_types]],
            # "args2" : ["visible", [trace.visible for trace in fig.data]], 
            "label": "Toggle All",
            "visible" : True, 

        }
buttons_airports.append(all_airports_button) 

## Show all flights button ##
flight_button = {
            "method": "restyle",
            # When toggled on, checkbox shows already visible traces + haunted place specified in box
            "args" : [{"visible" : True}, [i for i, x in enumerate(all_traces) if x.name == "Flights"]],
            # When toggled off, checkbox removes haunted trace
            "args2" : [{'visible':'legendonly'},[i for i,x in enumerate(all_traces) if x.name == "Flights"]],
            # "args2" : ["visible", [trace.visible for trace in fig.data]], 
            "label": "Flight Paths",
            "visible" : True, 

        }
buttons_airports.append(flight_button)



## Final Conf ##

updateMenusConf = [
    {
        "buttons": button_haunts,
        "direction" : "down",
        "showactive" : False,
        "x": 0.1,
        "y": 1.15,
        "xanchor" : "left",
        "yanchor" : "top", 
        "font": {"size" : 12},
        "type": "dropdown",
        "name": "Toggle Flags"
    },
    {
        "buttons": buttons_airports,
        "direction" : "down",
        "showactive" : True,
        "x": -.05,
        "y": 1.15,
        "xanchor" : "left",
        "yanchor" : "top", 
        "font": {"size" : 12},
        "type": "dropdown",
        "name": "Flight Toggle"
    }
    ]


## Create Plotly figure ##
fig = go.Figure(data = all_traces)


fig.update_layout(
    title_text = 'Flight Paths Accross U.S.',
    showlegend = True,
    clickmode='event+select',
    hovermode = 'closest',
    geo = dict(
        scope = 'north america',
        projection_type = 'azimuthal equal area',
        showland = True,
        showcountries = True,
        showsubunits = True, 
        subunitcolor = "Black",
        landcolor = 'rgb(243, 243, 243)',
        countrycolor = 'rgb(204, 204, 204)',
    ),
    updatemenus = updateMenusConf
)

# Remove html if it already exists 
if os.path.exists('airborne_events.html'):
    os.remove("airborne_events.html")

fig.write_html("airborne_events.html")


# Show plot
fig.show()



/var/folders/_b/bl5yw1q15msg5ky9z_04dpt40000gn/T/ipykernel_9755/718797123.py:22: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

/var/folders/_b/bl5yw1q15msg5ky9z_04dpt40000gn/T/ipykernel_9755/718797123.py:22: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

/var/folders/_b/bl5yw1q15msg5ky9z_04dpt40000gn/T/ipykernel_9755/718797123.py:22: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the docu